# 📊 Procesamiento Automático de Logs en Entornos Oracle JD Edwards

**Andrea Olcina** — Trabajo de Fin de Máster

---

Este cuaderno sirve como guía sobre los distintos módulos de la solución y su uso, así como de aquellos utilizados para el análisis de este Trabajo de Fin de Máster.

## 📑 Contenido

0. [Setup e instalación](#0)
1. [Parsing](#1)
2. [Drain](#2)
   - 2.1 [Ejecución](#2.1)
   - 2.2 [Benchmarking](#2.2)
3. [Template Generator](#3)
   - 3.1 [Extracción de templates](#3.1)
   - 3.2 [Agrupación semántica](#3.2)
   - 3.3 [Benchmark de agrupación semántica](#3.3)
   - 3.4 [Generación de regex](#3.4)
   - 3.5 [Validación de regex](#3.5)
4. [Limpieza de resultados de prueba](#4)

<a id='0'></a>
## 0️⃣ Setup e instalación

In [ ]:
%pip install -r requirements.txt

In [ ]:
import glob 
import json 
import re
import pandas as pd 
import os
import shutil

### 🗂️ Orígenes disponibles

El parámetro `origin` se utiliza en diversos módulos a lo largo de este notebook. Los valores disponibles son:

| Origen |
|---|
| `ais` |
| `alert_jde` |
| `bssv` |
| `e1root` |
| `jas` |
| `jde` |
| `listener` |

Hay dos conjuntos de datos disponibles:
- **`example_logs/`** — conjunto reducido de ficheros, usado a lo largo de este notebook.
- **`parsed_logs/`** — conjunto completo con el que se ha probado la solución.

<a id='1'></a>
## 1️⃣ Parsing

Este módulo ejecuta el pipeline de parsing. La carpeta de origen de logs se puede indicar como argumento.

```python
def parsing_execution(log_folder: str = "logs/", ingest_to_opensearch: bool = False,
                       output_folder: str = "parsed_logs",
                       enable_performance_tracking: bool = False,
                       results_output_format: str = "parquet"):
```

**Argumentos principales**

| Argumento | Descripción |
|---|---|
| `log_folder` | Carpeta local donde están los logs a procesar. Solo se usa si `execution_mode == "LOCAL"`; en S3 siempre se parte de la raíz del bucket. |
| `output_folder` | Carpeta donde se guardan los resultados parseados (default: `"parsed_logs"`). |
| `enable_performance_tracking` | Si `True`, mide tiempos por fase/origen y genera `performance_metrics.*` al finalizar. Desactivado por defecto para no añadir overhead en ejecuciones normales. |
| `results_output_format` | Formato de los ficheros de resultados parseados: `'parquet'` (default, eficiente para producción) o `'json'` (más legible, útil para debugging). |

In [ ]:
!python parsing/init.py --log-folder example_logs/ --output-folder example_parsed_logs/ --results-output-format json

In [ ]:
# Para reproducir los resultados exactos del TFM con el mismo conjunto de datos
# !python parsing/init.py --enable-performance-tracking

### 🔍 Inspección de resultados

In [ ]:
# Busca los ficheros parseados generados en example_logs/ (formato parquet)
parquet_files = glob.glob("example_parsed_logs/**/*.parquet", recursive=True)
print(f"Ficheros encontrados: {parquet_files}")

if parquet_files:
    df = pd.read_parquet(parquet_files[0])
    print(f"\nShape: {df.shape}")
    print(f"Columnas: {list(df.columns)}")
    df.head(10)
else:
    print("⚠️ No se encontraron ficheros .parquet, revisa la carpeta de salida real.")

In [ ]:
# Busca los ficheros parseados generados en example_logs/ (formato json)
json_files = glob.glob("example_parsed_logs/**/*.json", recursive=True)
print(f"Ficheros encontrados: {len(json_files)}")

if json_files:
    with open(json_files[4], encoding="utf-8") as f:
        original_data = json.load(f)
    print(original_data['records'][0])
else:
    print("⚠️ No se encontraron ficheros .json, revisa la carpeta de salida real.")

<a id='2'></a>
## 2️⃣ Drain

<a id='2.1'></a>
### 2.1 Ejecución

**Ejecutar Drain3 sobre logs ya parseados (`parsed3`), origen `example`:**
```bash
python drain/drain_unified.py parsed3 \
    --origin example \
    --parsed-folder parsed_logs \
    --output drain3_benchmarks_parsed_example \
    --depth 4 --st 0.4 --max-children 100 \
    --no-auto-update
```

Otras implementaciones disponibles:

| Comando | Descripción |
|---|---|
| `drain` | Drain (custom) sobre logs crudos |
| `parsed` | Drain sobre el contenido semántico del mensaje |
| `drain3` | Drain3 sobre logs crudos |
| `parsed3` | Drain3 sobre el contenido semántico del mensaje |

**Parámetros principales**

| Argumento | Descripción |
|---|---|
| `--origin` | Nombre del origen a procesar (debe existir `config/<origin>_config.py`). |
| `--parsed-folder` | Carpeta con los parquet ya parseados (patrón `YYYY_MM_DD_<origin>.parquet`). |
| `--parsed-format` | Formato de los logs parseados a leer (default: `parquet`) — `parquet` \| `json`. Solo para `parsed` y `parsed3`. |
| `--output` | Carpeta donde se guardará `<origin>_drain3_parsed_full.json`. |
| `--depth / --st / --max-children` | Hiperparámetros de Drain3 (si se omiten, se usa el perfil guardado en la config del origen). |
| `--no-auto-update` | Evita que Drain3 sobrescriba automáticamente `config/<origin>_config.py` con el "mejor perfil" detectado (recomendado durante benchmarking manual). |

In [ ]:
origin = "ais"

In [ ]:
!python drain/drain_unified.py parsed3 \
    --origin $origin \
    --parsed-folder example_parsed_logs \
    --parsed-format json \
    --output drain3_parsed_example \
    --no-auto-update

In [ ]:
with open(f"drain3_parsed_example/{origin}/{origin}_drain3_parsed_clusters.json", encoding="utf-8") as f:
    raw_data = json.load(f)

results = pd.DataFrame(raw_data["clusters"])
results

<a id='2.2'></a>
### 2.2 Benchmarking

Ejecuta y extrae una serie de métricas para medir la calidad del algoritmo elegido. Este módulo evalúa varias premisas:

- La capacidad de Drain (implementación personalizada) frente a Drain3.
- La capacidad de Drain como algoritmo sobre logs crudos o sobre logs ya procesados extrayendo el contenido semántico del mensaje.
- La mejor combinación de parámetros de `sim_th` y `depth`.

**Ejemplo — benchmark para origen `example`:**
```bash
python drain/benchmarking/drain_benchmarking_unified.py parsed3 \
    --origin example \
    --parsed-folder example_logs \
    --output drain3_benchmarks_parsed_example
```

Otras implementaciones disponibles: `drain`, `parsed`, `drain3` (mismo significado que en 2.1).

**Parámetros principales**

| Argumento | Descripción |
|---|---|
| `--origin` | Nombre del origen a benchmarkear. |
| `--all` | Si se activa, ejecuta el benchmark para **todos** los orígenes detectados en `--parsed-folder`, en vez de uno solo. |
| `--parsed-folder` | Carpeta con los logs ya parseados. |
| `--parsed-format` | Formato de los logs parseados a leer (default: `parquet`) — `parquet` \| `json`. Solo para `parsed` y `parsed3`. |
| `--output` | Carpeta donde se guardará `<origin>_drain3_parsed_full.json`. |

In [ ]:
origin = "ais"

In [ ]:
!python drain/benchmarking/drain_benchmarking_unified.py parsed3 \
    --origin $origin \
    --parsed-folder example_parsed_logs \
    --parsed-format json \
    --output drain3_benchmarks_parsed_example

In [ ]:
with open("drain3_benchmarks_parsed_example/ais_benchmark_parsed.json", encoding="utf-8") as f:
    raw_data = json.load(f)

results = pd.DataFrame(raw_data["results"])
results

<a id='3'></a>
## 3️⃣ Template Generator

Para la generación de templates, el primer paso es extraer los templates generados por Drain3 sobre los logs procesados, a una carpeta común.

> ⚠️ Si todavía no has ejecutado Drain3, vuelve a la [sección 2](#2) antes de continuar.

<a id='3.1'></a>
### 3.1 Extracción de templates

**Ejecutar extracción de templates, origen `example`:**
```bash
python template_generator/extract_templates.py \
    --origin jde \
    --input-folder drain3_parsed_results_example \
    --output-folder templates
```

**Parámetros principales**

| Argumento | Descripción |
|---|---|
| `--origin` | Nombre del origen a procesar (debe existir `config/<origin>_config.py`). |
| `--all` | Extrae todos los orígenes desde los resultados de Drain3. |
| `--input-folder` | Carpeta con los resultados de Drain3 (default: `drain3_parsed_results`). |
| `--output-folder` | Carpeta de salida para `<origin>_template.json` (default: `templates`). |

In [ ]:
origin = "ais"

In [ ]:
!python template_generator/extract_templates.py \
    --origin $origin \
    --input-folder drain3_parsed_example \
    --output-folder templates_example

<a id='3.2'></a>
### 3.2 Agrupación semántica

El siguiente paso es agrupar estos clusters siguiendo una técnica de clustering aglomerativo jerárquico basado en distancia coseno. Para medir esta similitud semántica, se generan embeddings sobre los resultados de Drain3.

**Ejecutar agrupación semántica, origen `example`:**
```bash
python template_generator/semantic_grouping.py \
    --input templates_example/origin.json \
    --threshold 0.6 \
    --output templates_example
```

**Parámetros principales**

| Argumento | Descripción |
|---|---|
| `--input` | Ruta a `x_template.json`. |
| `--threshold` | Distancia coseno máxima dentro de un grupo (default `0.6`; más bajo = grupos más estrictos). |
| `--output` | Ruta de salida (default: `x_semantic_groups.json`). |

In [ ]:
origin = "ais"

In [ ]:
!python template_generator/semantic_grouping.py \
    templates_example/{origin}_template.json

#### ✅ Validación: Drain3 clusters vs. Semantic grouping

In [ ]:
# --- Clusters originales de Drain3 (antes de agrupar) ---
with open("templates_example/ais_template.json", encoding="utf-8") as f:
    original_data = json.load(f)
original_clusters = original_data.get("clusters", original_data if isinstance(original_data, list) else [])
num_original = len(original_clusters)

# --- Resultado del agrupamiento semántico ---
with open("templates_example/ais_template_semantic_groups.json", encoding="utf-8") as f:
    grouped_data = json.load(f)
groups = grouped_data.get("groups", grouped_data if isinstance(grouped_data, list) else [])
num_groups = len(groups)

# --- Nº de templates originales cubiertos por los grupos (sanity check) ---
num_templates_in_groups = sum(
    len(g.get("templates", g.get("member_cluster_ids", [])))
    for g in groups
)

print("=" * 60)
print("📊 VALIDACIÓN: Drain3 clusters vs Semantic grouping")
print("=" * 60)
print(f"  Clusters originales (Drain3):         {num_original}")
print(f"  Grupos tras agrupamiento semántico:   {num_groups}")
print(f"  Templates cubiertos dentro de grupos: {num_templates_in_groups}")
if num_original:
    reduccion_pct = (1 - num_groups / num_original) * 100
    print(f"  Reducción:                            {num_original - num_groups} ({reduccion_pct:.1f}% menos grupos)")

if num_templates_in_groups != num_original:
    print(f"\n⚠️  Descuadre: {num_original} clusters originales vs "
          f"{num_templates_in_groups} templates cubiertos en los grupos "
          f"(revisa si algún cluster quedó fuera del agrupamiento).")
else:
    print("\n✅ Todos los clusters originales están cubiertos por los grupos.")

<a id='3.3'></a>
### 3.3 Benchmark de agrupación semántica

Se comparan dos métodos:

- **Moderno**: Sentence Transformers con el modelo `all-MiniLM-L6-v2`.
- **Clásico**: TF-IDF + Truncated SVD.

Para ambos métodos también se evalúa el umbral de distancia de clustering.

**Ejecutar benchmark de agrupación semántica, origen `example`:**
```bash
python template_generator/benchmark_semantic_grouping.py \
    --inputs templates_example/origin.json \
    --summary_csv resumen_example.csv
```

**Parámetros principales**

| Argumento | Descripción |
|---|---|
| `--inputs` | Ruta(s) a `x_template.json` (uno o varios orígenes). |
| `--csv` | Ruta opcional para exportar resultados detallados a CSV. |
| `--summary_csv` | Ruta opcional para exportar la tabla resumen entre orígenes a CSV. |

In [ ]:
origin = "ais"

In [ ]:
!python template_generator/benchmark_semantic_grouping.py \
    templates_example/{origin}_template.json \
    --csv resumen_{origin}_example.csv

In [ ]:
df_semantic_bench = pd.read_csv(f"resumen_{origin}_example.csv")
df_semantic_bench

<a id='3.4'></a>
### 3.4 Generación de regex

Una vez los clusters están agrupados, se generan expresiones regulares codificando las partes variables.

**Ejecutar la generación de expresiones regulares a partir de los grupos semánticos:**
```bash
python template_generator/generate_regex_from_groups.py \
    --input x_semantic_groups_template.json
```

**Parámetros principales**

| Argumento | Descripción |
|---|---|
| `--input` | Ruta a `x_template_semantic_groups.json`. |

In [ ]:
!python template_generator/generate_regex_from_groups.py \
    templates_example/ais_template_semantic_groups.json

<a id='3.5'></a>
### 3.5 Validación de regex — sanity check

Comprueba con `re.match` que cada regex captura correctamente su propio `representative_template`, para detectar regex rotas antes de darlas por buenas.

In [ ]:
origin = "ais"
regex_path = f"templates_example/{origin}_template_regex.json"

In [ ]:
with open(regex_path, encoding="utf-8") as f:
    regex_data = json.load(f)

groups = regex_data.get("groups", [])
print(f"📋 {len(groups)} grupo(s) con regex generada para '{origin}'\n")

fails = []
for g in groups:
    pattern = g.get("regex")
    sample = g.get("representative_template") or (g.get("templates") or [None])[0]
    if not pattern or not sample:
        continue
    try:
        match = re.match(pattern, sample)
    except re.error as e:
        fails.append((sample, pattern, f"regex inválida: {e}"))
        continue
    if not match:
        fails.append((sample, pattern, "no hace match con su propio template"))

if fails:
    print(f"❌ {len(fails)} regex con problemas:")
    for sample, pattern, reason in fails:
        print(f"  - {reason}\n    template: {sample}\n    regex:    {pattern}\n")
else:
    print("✅ Todas las regex hacen match con su template representativo")

<a id='4'></a>
## 4️⃣ Limpieza de resultados de prueba

> ⚠️ Ejecutar solo cuando ya no necesites los resultados de prueba. No toca `example_logs/` (carpeta de entrada) ni `config/<origin>_config.py`.

In [ ]:
folders_to_clean = [
    "drain3_benchmarks_parsed_example",
    "drain_origin_benchmarks_example",
    "drain3_benchmarks_example",
    "drain_benchmarks_parsed_example",
    f"drain3_parsed_results_{origin}_example",
    "drain3_parsed_example",
    "example_parsed_logs",
    "templates_example",
]
files_to_clean = glob.glob(f"templates_example/{origin}_template*.json") + [f"resumen_{origin}_example.csv"]

for folder in folders_to_clean:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"🗑️  Eliminada carpeta: {folder}")

for file in files_to_clean:
    if os.path.exists(file):
        os.remove(file)
        print(f"🗑️  Eliminado fichero: {file}")

print("\n✅ Limpieza completada")